In [ ]:
# Imports and settings
import google.generativeai as genai
import matplotlib.pyplot as plt
import numpy as np
import io
import os
from datetime import date
from IPython.display import Markdown
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import font
from tkinter.filedialog import askopenfilename
from tkcalendar import Calendar

In [ ]:
# Global variables
dates = {}
target_cal = 0
target_pro = 0
target_carb = 0
target_fat = 0
img = None
model = None
verify_user = False
verify_api = False

In [ ]:
# Initialize Gemini AI using API key
def init_gem(api_key):
    global model
    # First, try API key entered by user
    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-1.5-flash")
        model.generate_content("Hello")
        return
    except:
        # Second, check if API key is stored in Environment Variables
        try:
            genai.configure(api_key=os.environ["GEMINI_API_KEY"])
            model = genai.GenerativeModel("gemini-1.5-flash")
            model.generate_content("Hello")
            return
        except:
            raise Exception("Invalid API key")

In [ ]:
# Initialize target variables
def init_var(sex, weight, height, age):
    global target_cal
    global target_pro
    global target_carb
    global target_fat
    # Using Mifflin-St Jeor formula (using kg and cm), and moderately active activity level (1.55x)
    if sex == "male":
        target_cal = (10 * weight + 6.25 * height - 5 * age + 5) * 1.55
    else:
        target_cal = (10 * weight + 6.25 * height - 5 * age - 161) * 1.55
    # Using composition that calories should come from 25% protein, 50% carbohydrates, and 25% fat. Then convert to grams
    target_pro = (0.25 * target_cal) / 4
    target_carb = (0.5 * target_cal) / 4
    target_fat = (0.25 * target_cal) / 9
    # Require positive metrics
    if target_cal <= 0 or target_pro <= 0 or target_carb <= 0 or target_fat <= 0:
        raise Exception("Must be all positive values")

In [ ]:
# Function to open user-specified image
def choose_image():
    tk.Tk().withdraw()
    filename = askopenfilename()
    img = Image.open(filename)
    # Downsize image for efficiency
    img.thumbnail(
        (200, 200), Image.LANCZOS
    )  # LANCZOS is more intensive but results in better image quality
    return img

In [ ]:
# Get a response from Gemini
def prompt_gemini(information, img):
    # Build question (string)
    instructions = "Using the information provided, estimate the quantity, calories, protein, carbohydrates, and fat of each food item."
    rules = [
        "Be descriptive.",
        "If there are multiple of the same food, combine them. For example, write 2 buns instead of writing 1 bun twice.",
        "Write in this format: # of [food], # cal, # g protein, # g carbohydrate, # g fat.",
        "Give a name for the dish on the first line.",
        "Write each ingredient on a new line.",
        "Use metric measurements.",
        "Round to the nearest whole number.",
        "Do not ask the user for more information.",
        "Do not write a disclaimer.",
        "Do not write a total.",
    ]
    string = (
        "Information: " + information + "\nInstructions: " + instructions + "\nRules:"
    )
    for i in rules:
        string += " " + i

    # Ask Gemini
    if img is None:
        return model.generate_content(string).text
    else:
        return model.generate_content([string, img]).text

In [ ]:
# Class to store and track data for a day
class Entry:
    def __init__(self):
        # Copy global targets into a field in case globals change in the future
        self.target_cal = target_cal
        self.target_pro = target_pro
        self.target_carb = target_carb
        self.target_fat = target_fat
        self.current_cal = 0
        self.current_pro = 0
        self.current_carb = 0
        self.current_fat = 0
        self.meals = []

    # Returns the progress of the current metrics to the target as a percentage, using all meals combined
    def get_total_progress(self):
        return [
            self.current_cal / self.target_cal * 100,
            self.current_pro / self.target_pro * 100,
            self.current_carb / self.target_carb * 100,
            self.current_fat / self.target_fat * 100,
        ]

    # Returns the progress of the current metrics to the target as a percentage, showing the contribution of each meal
    def get_split_progress(self):
        arr = []
        for meal in self.meals:
            arr.append(
                [
                    meal.get_cal() / self.target_cal * 100,
                    meal.get_pro() / self.target_pro * 100,
                    meal.get_carb() / self.target_carb * 100,
                    meal.get_fat() / self.target_fat * 100,
                ]
            )
        return arr

    # Returns the names of all stored meals
    def get_meal_names(self):
        arr = []
        for meal in self.meals:
            arr.append(meal.get_name())
        return arr

    # Add a meal to the array and update metrics
    def update(self, meal):
        self.meals.append(meal)
        self.current_cal += meal.get_cal()
        self.current_pro += meal.get_pro()
        self.current_carb += meal.get_carb()
        self.current_fat += meal.get_fat()

    # Print all meals
    def __str__(self):
        string = ""
        for meal in self.meals:
            string += str(meal)
        return string

In [ ]:
# Class to consolodate ingredients (items) into a meal
class Meal:
    def __init__(self, name, item, cal, pro, carb, fat):
        self.name = name
        self.item = item
        self.cal = cal
        self.pro = pro
        self.carb = carb
        self.fat = fat

    # Returns only the name of the meal
    def get_name(self):
        return self.name

    # Getter functions to sum all the ingredients' metrics
    def get_cal(self):
        return sum(self.cal)

    def get_pro(self):
        return sum(self.pro)

    def get_carb(self):
        return sum(self.carb)

    def get_fat(self):
        return sum(self.fat)

    # Print meal name with ingredients and their metrics
    def __str__(self):
        string = self.name + "\n"
        for i in range(len(self.item)):
            string += (
                "\t"
                + str(self.item[i])
                + ", "
                + str(self.cal[i])
                + " cal, "
                + str(self.pro[i])
                + " g, "
                + str(self.carb[i])
                + " g, "
                + str(self.fat[i])
                + " g\n"
            )
        string += (
            "\tTotal: "
            + str(self.get_cal())
            + " cal, "
            + str(self.get_pro())
            + " g protein, "
            + str(self.get_carb())
            + " g carbohydrates, "
            + str(self.get_fat())
            + " g fat\n"
        )
        return string

In [ ]:
# Parse response
def parse_response(text):
    split = text.split("\n")
    item = []
    cal = []
    pro = []
    carb = []
    fat = []
    # First line is the meal name
    name = split[0]
    # Loop through one ingredient (item) at a time
    for i in split[1:]:
        # Tokenize the line to seperate each metric
        line = i.split(",")
        # Filter empty lines
        if len(line) == 1:
            continue
        # Add so that an index corresponds to the same ingredient (item) across arrays
        item.append(line[0])
        cal.append(int(line[1].split()[0]))  # The number is always the first token
        pro.append(int(line[2].split()[0]))
        carb.append(int(line[3].split()[0]))
        fat.append(int(line[4].split()[0]))

    # Create meal object
    meal = Meal(name, item, cal, pro, carb, fat)
    # Store the meal
    today = str(date.today())
    if today not in dates:
        dates[today] = Entry()
    dates[today].update(meal)

    return meal

In [ ]:
# Helper function to print all stored meals
def print_all_meals():
    for date in dates:
        print(date)
        print(dates[date])

In [ ]:
# Function to graph the progress on the different metrics for a given day
def graph_progress(date):
    # Plot initial bar chart
    if date not in dates:
        raise Exception("No meals for this date")
    x = ["Calories", "Protein", "Carbohydrates", "Fat"]
    y = dates[date].get_split_progress()
    plt.bar(x, y[0])
    # Plot stacked bars
    bot = y[0]
    for i in range(1, len(y)):
        plt.bar(x, y[i], bottom=bot)
        bot = np.add(bot, y[i])
    plt.legend(
        dates[date].get_meal_names(), bbox_to_anchor=(1.02, 1)
    )  # Position legend to the top-right of plot
    # Threshold line
    plt.axhline(y=100, color="r")
    # Labels
    plt.xlabel("Metric")
    plt.ylabel("Percentage")
    plt.title("Goals for " + str(date))

    # Convert graph to image
    buffer = io.BytesIO()
    fig = plt.gcf()
    fig.savefig(buffer, bbox_inches="tight")
    buffer.seek(0)
    img = Image.open(buffer)
    plt.close()  # Prevents colours changing if graph is redrawn
    return img

In [ ]:
# Initialize UI
root = tk.Tk()
tk.Tk().withdraw()
root.title("Nutrition Tracker App")
root.geometry("1000x750")

# Define fonts
default_font = font.nametofont("TkDefaultFont")
default_font.configure(family="Arial", size=10)
bold_font = font.Font(root, family="Arial", size=10, weight="bold")
underline_font = font.Font(root, family="Arial", size=10, underline=True)

# Initialize Pages
settings_page = tk.Frame(root)
add_page = tk.Frame(root)
view_page = tk.Frame(root)
pages = [settings_page, add_page, view_page]

# UI Methods (Commands)
# Exit GUI
def exit():
    root.quit()  # Found that it is more consistent using both quit and destroy methods
    root.destroy()

# Show a given page
def show_page(index):
    # Hide all pages
    for page in pages:
        page.pack_forget()
    # Show the specified page
    pages[index].pack()
    # Special case - reset image for add meal page
    if index == 1:
        global img
        img = None

# Use user-given API key to initialize Gemini AI
def set_key(api_key):
    try:
        init_gem(api_key)
        global verify_api
        verify_api = True
        confirmation_label.config(fg="green", text="Success! API key has been set")
    except:
        confirmation_label.config(text="Failure. Please check your API key")

# Use user information to initialize target variables
def save(sex, weight, height, age):
    try:
        init_var(sex, weight, height, age)
        global verify_user
        verify_user = True
        confirmation_label.config(
            fg="green",
            text="Success!\n\nYour targets:\nCalories: "
            + str(round(target_cal, 2))
            + " cal\nProtein: "
            + str(round(target_pro, 2))
            + " g\nCarbohydrates: "
            + str(round(target_carb, 2))
            + " g\nFat: "
            + str(round(target_fat, 2))
            + " g",
        )
    except:
        confirmation_label.config(text="Failure. Please check your inputs/measurements")

# Set image and convert it for displaying
def choose_image_wrapper():
    global img
    img = choose_image()
    display_img = img.copy()
    display_img.thumbnail((100, 100), Image.LANCZOS)
    photo_img = ImageTk.PhotoImage(display_img, master=root)
    image_label.config(image=photo_img)
    image_label.image = photo_img  # Needed to prevent blank image

# Add a meal to the food diary
def add(information, img):
    # Check that API key has been set
    if verify_api is False:
        error_label.config(text="Please set your API key first")
        return
    # Check that targets have been set
    if verify_user is False:
        error_label.config(text="Please set your user details first")
        return
    # Check that input is not blank
    information = information.strip()
    if information == "" and img is None:
        error_label.config(text="No text or image detected")
    # Ask gemini
    else:
        text = prompt_gemini(information, img)
        try:
            meal = parse_response(text)
            error_label.config(fg="green", text="Success!\n\nYour meal:\n" + str(meal))
        except:
            error_label.config(text="AI generation error. Please try again")

# Open a pop-up window with a calendar
def show_calendar():

    # Use the calendar input to display the correct graph
    def show_graph():
        date = calendar.selection_get()
        try:
            img = graph_progress(str(date))
            display(img)
            photo_img = ImageTk.PhotoImage(img, master=root)
            graph_label.config(image=photo_img)
            graph_label.image = photo_img
            top.destroy()
        except:
            feedback_label.config(text="No stored meals for " + str(date))

    top = tk.Toplevel(root)
    today = date.today()
    calendar = Calendar(
        top,
        selectmode="day",
        year=int(today.strftime("%Y")),
        month=int(today.strftime("%m").lstrip("0")),
        day=int(today.strftime("%d").lstrip("0")),
    )
    calendar.pack()
    select_button = tk.Button(top, text="Select", command=show_graph)
    select_button.pack()
    feedback_label = tk.Label(top, fg="red")
    feedback_label.pack()

# Initialize Menu
menubar = tk.Menu(root)
root.config(menu=menubar)
filemenu = tk.Menu(menubar, tearoff=0)
menubar.add_cascade(label="File", menu=filemenu)
filemenu.add_command(label="Exit", command=exit)
navmenu = tk.Menu(menubar, tearoff=0)
menubar.add_cascade(label="Navigate", menu=navmenu)
navmenu.add_command(label="Settings", command=lambda: show_page(0))
navmenu.add_command(label="Log Meal", command=lambda: show_page(1))
navmenu.add_command(label="View Diary", command=lambda: show_page(2))

# User information page
api_label = tk.Label(settings_page, text="API Key:", font=bold_font)
api_label.pack()
api_text = tk.Text(settings_page, width=40, height=1)
api_text.pack()
api_button = tk.Button(
    settings_page,
    text="Verify Key",
    font=bold_font,
    command=lambda: set_key(api_text.get("1.0", "end-1c").strip()),
)  # 1.0 is line 0 character 1, end-1c is the end minus the \n
api_button.pack(pady=20)
settings_label = tk.Label(settings_page, text="User details:", font=bold_font)
settings_label.pack()
sex_label = tk.Label(settings_page, text="Sex")
sex_label.pack()
sex_options = ["Male", "Female"]
sex_var = tk.StringVar(settings_page, value=sex_options[0])
sex_selector = tk.OptionMenu(settings_page, sex_var, *sex_options)
sex_selector.pack()
weight_scale = tk.Scale(
    settings_page,
    from_=0,
    to=250,
    orient=tk.HORIZONTAL,
    length=200,
    label="Weight (kg)",
)
weight_scale.pack()
height_scale = tk.Scale(
    settings_page,
    from_=0,
    to=250,
    orient=tk.HORIZONTAL,
    length=200,
    label="Height (cm)",
)
height_scale.pack()
age_scale = tk.Scale(
    settings_page,
    from_=0,
    to=100,
    orient=tk.HORIZONTAL,
    length=200,
    label="Age (years)",
)
age_scale.pack()
save_button = tk.Button(
    settings_page,
    text="Save Details",
    font=bold_font,
    command=lambda: save(
        sex_var.get(), weight_scale.get(), height_scale.get(), age_scale.get()
    ),
)
save_button.pack(pady=20)
confirmation_label = tk.Label(settings_page, fg="red")
confirmation_label.pack()

# Add meal page
add_label = tk.Label(add_page, text="Add a meal:", font=bold_font)
add_label.pack()
instructions_label = tk.Label(
    add_page, text="Describe the meal using text and/or an image\n\nText Description"
)
instructions_label.pack()
info_text = tk.Text(add_page, width=40, height=3)
info_text.pack()
upload_button = tk.Button(add_page, text="Upload Image", command=choose_image_wrapper)
upload_button.pack(pady=20)
image_label = tk.Label(add_page, text="(No image chosen)")
image_label.pack()
add_button = tk.Button(
    add_page,
    text="Add Meal",
    font=bold_font,
    command=lambda: add(info_text.get("1.0", "end-1c").strip(), img),
)
add_button.pack(pady=20)
error_label = tk.Label(add_page, fg="red", justify=tk.LEFT)
error_label.pack()

# View diary page
view_label = tk.Label(view_page, text="View food diary:", font=bold_font)
view_label.pack()
calendar_button = tk.Button(view_page, text="Choose a date", command=show_calendar)
calendar_button.pack(pady=20)
graph_label = tk.Label(view_page)
graph_label.pack()

# Run
show_page(0)
root.mainloop()